In [2]:
import numpy as np
import os
from tqdm import tqdm
import h5py

In [3]:
target_folder="../data/prottrans/all/Oth/test"
database_dataset="../RAG/dataset/prottrans/CDNO_hybrid.npy"
path_output="./rag_prottrans/all/Oth/test.npy"

In [4]:
database=np.load(database_dataset)

In [5]:
def cal_length(embeddinng, maxseq ,num_feature):
    data = np.zeros((maxseq, num_feature), dtype=np.float64)
    data_len = len(embeddinng)
    if data_len < maxseq:
        data[:data_len, :] = embeddinng
    else:
        data[:, :] = embeddinng[:maxseq, :]
    #print(data.shape)
    data = data.reshape((1, 1, maxseq, num_feature))    
    return data

In [6]:
def vector_distance(target,dataset,maxseq,num_feature):
    target_len=cal_length(target,maxseq,num_feature)
    dist_tmp=[]
    new_embedding=[]
    for i in dataset:
        dist = np.linalg.norm(target_len - i)
        if len(dist_tmp) < 5:
            dist_tmp.append(dist)
            new_embedding.append(i)
            if len(dist_tmp) == 5:
                sorted_indices = np.argsort(dist_tmp)
                dist_tmp = [dist_tmp[j] for j in sorted_indices]
                new_embedding = [new_embedding[j] for j in sorted_indices]
        elif dist < dist_tmp[-1]:
            dist_tmp[-1] = dist
            new_embedding[-1] = i
            sorted_indices = np.argsort(dist_tmp)
            dist_tmp = [dist_tmp[j] for j in sorted_indices]
            new_embedding = [new_embedding[j] for j in sorted_indices]
    avg_embedding = np.mean(new_embedding, axis=0)
    combined_embedding = (target_len + avg_embedding) / 2
    #print("     avg_embedding:",np.linalg.norm(target_len - avg_embedding))
    #print("combined_embedding:",np.linalg.norm(target_len - combined_embedding))
    #vali(target_len,new_embedding)
    return combined_embedding

In [7]:
def vali(target_len,new_embedding):
    diss=[]
    for i in new_embedding:
        diss.append(np.linalg.norm(target_len - i))
    print(diss)

In [8]:
def saveData(path,data):
    #data= data[:, np.newaxis, :, :]
    print(data.shape)
    np.save(path, data)

In [9]:
input_dir=os.listdir(target_folder)
result=[]
maxseq=35
num_feature=1024
for i in tqdm(input_dir, desc="Processing files"):
    if i.endswith(".prottrans"):
        data = np.loadtxt(os.path.join(target_folder, i))
        combine = vector_distance(data, database, maxseq, num_feature)
        result.append(combine)
data = np.concatenate(result, axis=0)
saveData(path_output, data)

Processing files: 100%|████████████████████████████████████████████████████████████| 62/62 [00:58<00:00,  1.06it/s]

(62, 1, 35, 1024)
